In [ ]:
# =============================================================================
# jobs/26_fgs_forecast_verification.py
# Did the forecast risk for a constituency line up with the flood alerts and warnings
# that were actually in force?
#
# One row per constituency, forecast date and LEAD DAY (1 to 5: the forecast made that
# many days before, lead 1 being the statement issued on the day itself) where something
# was forecast or something was in force. It holds the forecast rank (the worst across
# flood sources) and the number of areas with an alert, a warning and a severe flood
# warning in force on that date. The app's Forecast verification tab reads it.
#
# What this script does:
#   1. Takes the LAST statement issued on each date (an amendment replaces the morning
#      issue), as 21 does. Its days 1 to 5 are the forecasts for five dates, at leads
#      1 to 5. Polygons covering less than FGS_CONTACT_MIN_PCT of a constituency are
#      ignored. A statement with no flagged area still counts as coverage, so a quiet
#      forecast is not mistaken for a missing one.
#   2. Reads ea_warning_episodes and ea_warning_periods (job 25) and the areas linked to
#      each constituency (ea_area_constituency_lookup, at WARNING_CONTACT_MIN_PCT, the same
#      rule as the warning contacts and the app's Warning history).
#   3. Works out, for every date up to yesterday, how many areas were in force in each
#      constituency (utils/verification.py has the rules): any point in the date counts, an
#      area counts once however often it was issued, and an episode open for over 30 days
#      counts only on its first date.
#   4. Writes fgs_forecast_verification in full.
#
# SPARSE ON PURPOSE. A date with nothing forecast and nothing in force has no row. That
# gives hits, false alarms and misses, but not "correctly stayed quiet", so no measure
# here can say how often a quiet forecast was right.
#
# Alerts and warnings are the Environment Agency's decisions, not observed flooding.
#
# Run after 25 (Master_polling keeps it current). Exit values: "success".
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from datetime import date, timedelta

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

from config import (
    TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_FGS_CONST_INTERSECT, TBL_FGS_VERIFICATION,
    TBL_WARNING_EPISODES, TBL_WARNING_PERIODS, TBL_EA_CONST_LOOKUP,
    FGS_CONTACT_MIN_PCT, WARNING_CONTACT_MIN_PCT, FGS_GEOMETRY_SOURCE
)
from utils.helpers import get_spark, utc_now, table_exists, source_predicate, risk_rank_sql
from utils.verification import (
    spans_from_episodes, daily_in_force, build_verification, LEADS
)


# =============================================================================
# SETUP
# =============================================================================

spark = get_spark()
now   = utc_now()

for table in (TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_FGS_CONST_INTERSECT,
              TBL_WARNING_EPISODES, TBL_WARNING_PERIODS, TBL_EA_CONST_LOOKUP):
    if not table_exists(spark, table):
        raise Exception(f"{table} does not exist. Run 01/02, 06, 07 and 25 first.")

LAST_DAY = (now - timedelta(days=1)).date()     # today is still under way, so it is not scored
print(f"FGS_GEOMETRY_SOURCE = {FGS_GEOMETRY_SOURCE}; scoring dates up to {LAST_DAY}")


In [ ]:
# =============================================================================
# THE FORECAST SIDE: what was forecast, and for which dates a statement exists
# =============================================================================

l_in_source = source_predicate(spark, TBL_FGS_INTERSECT_LOG, FGS_GEOMETRY_SOURCE, alias="l")
i_in_source = source_predicate(spark, TBL_FGS_CONST_INTERSECT, FGS_GEOMETRY_SOURCE, alias="i")

FINAL_STATEMENTS = f"""
    SELECT statement_id, issue_date FROM (
        SELECT s.statement_id,
               to_date(s.issued_at) AS issue_date,
               ROW_NUMBER() OVER (
                   PARTITION BY to_date(s.issued_at)
                   ORDER BY s.issued_at DESC, s.statement_id DESC
               ) AS rn
        FROM {TBL_FGS_INTERSECT_LOG} l
        JOIN {TBL_STATEMENTS} s
          ON s.statement_id = l.statement_id
         AND COALESCE(s.last_modified_at, '') = COALESCE(l.last_modified_at, '')
        WHERE {l_in_source}
    ) WHERE rn = 1
"""

forecast = spark.sql(f"""
    WITH final AS ({FINAL_STATEMENTS})
    SELECT i.constituency_id,
           to_date(i.forecast_date)               AS forecast_date,
           i.day_index                            AS lead_day,
           MAX({risk_rank_sql('i.risk_level')})   AS forecast_rank
    FROM {TBL_FGS_CONST_INTERSECT} i
    JOIN final f ON f.statement_id = i.statement_id
    WHERE {i_in_source}
      AND i.day_index BETWEEN 1 AND 5
      AND i.forecast_date IS NOT NULL
      AND (i.intersection_pct IS NULL OR i.intersection_pct >= {FGS_CONTACT_MIN_PCT})
    GROUP BY i.constituency_id, to_date(i.forecast_date), i.day_index
    HAVING MAX({risk_rank_sql('i.risk_level')}) >= 1
""").toPandas()

# The date each final statement's day 1 falls on (its issue date when it flagged nothing), then
# the five dates it covers, one per lead.
first_days = spark.sql(f"""
    WITH final AS ({FINAL_STATEMENTS})
    SELECT f.statement_id,
           COALESCE(MIN(CASE WHEN i.day_index = 1 THEN to_date(i.forecast_date) END), f.issue_date) AS d1
    FROM final f
    LEFT JOIN {TBL_FGS_CONST_INTERSECT} i
      ON i.statement_id = f.statement_id AND {i_in_source}
    GROUP BY f.statement_id, f.issue_date
""").toPandas()
d1 = pd.to_datetime(first_days["d1"])
coverage = pd.concat(
    [pd.DataFrame({"forecast_date": d1 + pd.Timedelta(days=k - 1), "lead_day": k}) for k in LEADS],
    ignore_index=True)

print(f"Final statements: {len(first_days):,}   forecasts of rank 1+: {len(forecast):,}   "
      f"dates covered: {coverage['forecast_date'].nunique():,} "
      f"({coverage['forecast_date'].min():%Y-%m-%d} to {coverage['forecast_date'].max():%Y-%m-%d})")


# =============================================================================
# THE OUTCOME SIDE: alerts and warnings in force, by constituency and date
# =============================================================================

episodes = spark.table(TBL_WARNING_EPISODES).select(
    "episode_id", "ea_area_code", "area_type", "start_time", "end_time", "stale_open").toPandas()
periods = spark.table(TBL_WARNING_PERIODS).select(
    "episode_id", "ea_area_code", "level", "start_time", "end_time").toPandas()
links = spark.sql(f"""
    SELECT DISTINCT ea_area_code, constituency_id
    FROM {TBL_EA_CONST_LOOKUP}
    WHERE intersection_pct >= {WARNING_CONTACT_MIN_PCT}
""").toPandas()

spans    = spans_from_episodes(episodes, periods, now.replace(tzinfo=None))
outcomes = daily_in_force(spans, links, LAST_DAY)
print(f"Episodes: {len(episodes):,}   linked areas: {len(links):,}   "
      f"constituency-dates with something in force: {len(outcomes):,}")

result = build_verification(forecast, coverage, outcomes, LAST_DAY)
result["computed_at"] = now.isoformat()
print(f"Verification rows: {len(result):,}")
if len(result) == 0:
    raise Exception("No verification rows: check that 07 and 25 have run and that the lookup is not empty.")


# =============================================================================
# A QUICK READ: Medium or above against an alert or warning, by lead day
# =============================================================================

r = result.assign(forecast_hit=result["forecast_rank"] >= 3, happened=(result["alerts"] + result["warnings"]) > 0)
rows = []
for k in LEADS:
    x = r[r["lead_day"] == k]
    hits = int((x["forecast_hit"] & x["happened"]).sum())
    false_alarms = int((x["forecast_hit"] & ~x["happened"]).sum())
    misses = int((~x["forecast_hit"] & x["happened"]).sum())
    rows.append({"lead": k, "hits": hits, "false_alarms": false_alarms, "misses": misses,
                 "hit_rate": round(hits / (hits + misses), 3) if hits + misses else None,
                 "precision": round(hits / (hits + false_alarms), 3) if hits + false_alarms else None})
print("\nForecast Medium or above against any alert or warning in force, by lead day:")
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# =============================================================================
# WRITE TO DELTA
# Full overwrite: cheap to rebuild, and an amended statement or a late warning message
# changes the past.
# =============================================================================

SCHEMA = StructType([
    StructField("constituency_id", StringType(),  True),
    StructField("forecast_date",   DateType(),    True),
    StructField("lead_day",        IntegerType(), True),
    StructField("forecast_rank",   IntegerType(), True),
    StructField("alerts",          IntegerType(), True),
    StructField("warnings",        IntegerType(), True),
    StructField("severe",          IntegerType(), True),
    StructField("computed_at",     StringType(),  True),
])

out = result.copy()
out["forecast_date"] = pd.to_datetime(out["forecast_date"]).dt.date
for c in ("lead_day", "forecast_rank", "alerts", "warnings", "severe"):
    out[c] = out[c].astype("int32")

(
    spark.createDataFrame(out, SCHEMA).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TBL_FGS_VERIFICATION)
)

spark.sql(f"""
    COMMENT ON TABLE {TBL_FGS_VERIFICATION} IS
    'The FGS forecast risk for a constituency against the flood alerts and warnings in force.
     One row per constituency, forecast date and lead day (1 = the statement issued on the day,
     5 = four days before) where something was forecast or in force. forecast_rank is the worst
     across flood sources (1 Very Low, 2 Low, 3 Medium, 4 High, 0 none). alerts, warnings and
     severe count the areas in force at any point on that date; an area counts once however
     often it was issued. Sparse: a date with nothing forecast and nothing in force has no row,
     so hits, false alarms and misses can be counted but correct quiet forecasts cannot.
     Dates up to yesterday only. Alerts and warnings are EA decisions, not observed flooding.
     Rebuilt daily by 26_fgs_forecast_verification.'
""")

written = spark.table(TBL_FGS_VERIFICATION).count()
print(f"\nVerified: {written:,} rows written to {TBL_FGS_VERIFICATION}.")

dbutils.notebook.exit("success")
